# Drone Swarm Robotics - v2 extensions
Quadrotor dynamics & wind, communication loss/latency, moving obstacles, multiple swarms, coverage control with batteries, consensus, task allocation, learned controller gains and real-drone export.

Runs in Colab: *Runtime > Run all*.

In [ ]:
!pip -q install git+https://github.com/guptuv/drone-swarm-robotics.git
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib import rc
from IPython.display import HTML, display
from drone_swarm import *
from drone_swarm.coverage import CoverageSwarm
from drone_swarm.consensus import run_consensus, leader_follower
from drone_swarm.allocation import auction_allocation, greedy_nearest, random_allocation, evaluate
from drone_swarm.learning import evolve, evaluate as es_evaluate, PARAMS
rc('animation', html='jshtml'); plt.rcParams['animation.embed_limit'] = 80

## 1. Quadrotor dynamics + wind gusts
Thrust/tilt limits and attitude lag instead of a point mass.

In [ ]:
GOAL = np.array([80., 75, 20]); CITY = make_city(seed=3, n=7)
sw = DroneSwarm(SwarmConfig(n_drones=25), start_center=(12, 12, 10), seed=0)
sw.buildings, sw.goal = CITY, GOAL
sw.dynamics = QuadrotorDynamics(25, tilt_max_deg=35)
sw.wind = Wind(mean=(4, 2, 0), gust_std=2.0)
sw.run(900, phase='navigate (quadrotor + wind)', until=lambda s: np.linalg.norm(s.pos.mean(0) - GOAL) < 6)
fig, summary = plot_metrics(sw, 'Quadrotor + wind'); plt.show(); display(summary)
HTML(animate_swarm(sw, every=5, goals=[GOAL], title='Quadrotor + wind').to_jshtml())

## 2. Communication: packet loss and latency

In [ ]:
rows = []
for p_loss in [0, 0.6, 0.9]:
    for delay in [0, 10, 20]:
        s = DroneSwarm(SwarmConfig(n_drones=30), start_center=(50, 50, 25), spread=15, seed=0)
        s.vel = s.rng.normal(0, 4, (30, 3)); s.comms = CommModel(30, range_m=20, p_loss=p_loss, delay_steps=delay)
        s.run(400); d = s.df().iloc[100:]
        rows.append(dict(p_loss=p_loss, delay_s=delay * 0.1, polarization=d.polarization.mean(), spread_m=d.cohesion.mean(), delivery=s.comms.delivery_ratio()))
pd.DataFrame(rows).round(2)

## 3. Moving obstacles + two swarms crossing

In [ ]:
sw = DroneSwarm(SwarmConfig(n_drones=25), start_center=(12, 12, 20), seed=3)
sw.goal = np.array([85., 85, 25]); sw.moving_obstacles = random_obstacles(5, speed=6, seed=13)
sw.run(600, phase='fly to goal', until=lambda s: np.linalg.norm(s.pos.mean(0) - s.goal) < 6)
print('obstacle hits:', sw.obstacle_hits, '| min clearance:', round(sw.df().iloc[1:].min_obstacle_dist.min(), 2), 'm')

n = 30; cross = DroneSwarm(SwarmConfig(n_drones=n), seed=1); rng = np.random.default_rng(0)
cross.team = np.r_[np.zeros(15, int), np.ones(15, int)]
cross.pos[:15] = np.c_[rng.uniform(8, 20, 15), rng.uniform(35, 65, 15), rng.uniform(20, 30, 15)]
cross.pos[15:] = np.c_[rng.uniform(80, 92, 15), rng.uniform(35, 65, 15), rng.uniform(20, 30, 15)]
cross.vel[:] = 0; cross.traj[0] = cross.pos.copy()
G = np.zeros((n, 3)); G[:15] = [88, 50, 25]; G[15:] = [12, 50, 25]; cross.goal = G
cross.run(220, phase='two swarms swap sides')
print('min distance between teams:', round(cross.df().iloc[1:].min_cross_team.min(), 2), 'm')
HTML(animate_swarm(cross, every=4, rotate=False, title='Two swarms crossing').to_jshtml())

## 4. Coverage control (Lloyd / Voronoi) with battery return-to-charge

In [ ]:
cov = CoverageSwarm(n=10, seed=0).run(120)
bat = CoverageSwarm(n=10, seed=0, battery=True).run(600)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].imshow(cov.owner_hist[-1].reshape(cov.gx.shape), origin='lower', extent=(0, 100, 0, 100), cmap='tab10', alpha=.35)
ax[0].contour(cov.gx, cov.gy, cov.phi, levels=6, colors='k', alpha=.3); ax[0].scatter(*cov.traj[-1].T, c='tab:blue', edgecolors='k'); ax[0].set_title('Voronoi cells')
ax[1].semilogy(cov.cost); ax[1].set_title('Locational cost')
ax[2].plot(bat.active_hist, label='drones covering'); ax[2].plot(np.array(bat.batt_hist).min(1) / 10, label='min battery / 10'); ax[2].legend(); ax[2].set_title('Battery-aware coverage')
plt.show()

## 5. Consensus: rendezvous needs a connected graph

In [ ]:
rng = np.random.default_rng(1); p0 = rng.uniform(0, 60, (12, 2))
for R in [15, 25, 35, 45]:
    _, dis, lam = run_consensus(p0, comm_range=R)
    plt.semilogy(np.maximum(dis, 1e-4), label=f'range {R} m, lambda2={max(lam[0], 0):.2f}')
plt.legend(); plt.xlabel('iteration'); plt.ylabel('disagreement (m)'); plt.show()

## 6. Task allocation: auction vs greedy vs random

In [ ]:
r = np.random.default_rng(3); starts, tasks = r.uniform(0, 10, (5, 2)), r.uniform(0, 100, (30, 2))
pd.DataFrame({name: evaluate(starts, tasks, fn(starts, tasks)) for name, fn in [('auction', auction_allocation), ('greedy', greedy_nearest), ('random', random_allocation)]}).T.round(1)

## 7. Learning controller gains with an Evolution Strategy (~1 min)

In [ ]:
c = SwarmConfig(); theta0 = [c.w_sep, c.w_ali, c.w_coh, c.w_goal, c.w_obs, c.separation_radius]
best, hist = evolve(theta0, generations=8, popsize=8)
for name, th in [('default', theta0), ('learned', best)]:
    J, det = es_evaluate(th, seeds=tuple(range(10, 18)), return_details=True)
    print(name, round(J, 1), pd.DataFrame(det)[['time', 'near_misses', 'building_hits', 'min_sep']].mean().round(2).to_dict())

## 8. Export for real drones (CSV, JSON, QGroundControl .plan)

In [ ]:
from drone_swarm.export import to_csv, to_qgc_plans, to_waypoint_json
sw = DroneSwarm(SwarmConfig(n_drones=5)); sw.set_formation('circle', center=(40, 40, 15), radius=10); sw.run(200)
to_csv(sw, 'trajectories.csv', every=5); to_waypoint_json(sw, 'waypoints.json')
print(to_qgc_plans(sw, 'plans', home_lat=28.6139, home_lon=77.2090))